**Calculation embeddings Uni-MOL and usage xgb**

In [2]:
from unimol_tools import MolTrain, MolPredict

In [3]:
import pandas as pd
import numpy as np
import torch

In [4]:
from unimol_tools import UniMolRepr
# single smiles unimol representation
clf = UniMolRepr(data_type='molecule', 
                 remove_hs=False,
                 model_name='unimolv1', # avaliable: unimolv1, unimolv2
                 model_size='84m', # work when model_name is unimolv2. avaliable: 84m, 164m, 310m, 570m, 1.1B.
                 )
smiles = 'c1ccc(cc1)C2=NCC(=O)Nc3c2cc(cc3)[N+](=O)[O]'
smiles_list = [smiles]
unimol_repr = clf.get_repr(smiles_list, return_atomic_reprs=True)


2025-04-18 16:17:49 | unimol_tools/models/unimol.py | 135 | INFO | Uni-Mol Tools | Loading pretrained weights from /opt/anaconda/envs/torch_gcnn/lib/python3.10/site-packages/unimol_tools/weights/mol_pre_all_h_220816.pt
2025-04-18 16:17:51 | unimol_tools/data/conformer.py | 150 | INFO | Uni-Mol Tools | Start generating conformers...
1it [00:00,  5.70it/s]
2025-04-18 16:17:51 | unimol_tools/data/conformer.py | 161 | INFO | Uni-Mol Tools | Succeeded in generating conformers for 100.00% of molecules.
2025-04-18 16:17:51 | unimol_tools/data/conformer.py | 178 | INFO | Uni-Mol Tools | Succeeded in generating 3d conformers for 100.00% of molecules.
2025-04-18 16:17:51 | unimol_tools/tasks/trainer.py | 77 | INFO | Uni-Mol Tools | Number of GPUs available: 1
2025-04-18 16:17:51 | unimol_tools/tasks/trainer.py | 97 | INFO | Uni-Mol Tools | Using single GPU.
100%|██████████| 1/1 [00:01<00:00,  1.32s/it]


In [5]:
df = pd.read_csv('df.csv')
df

,MP,SMILES
0,355.15,O=[N+]([O-])c1c(O)cccc1O
1,373.65,Cc1c([N+](=O)[O-])cc([N+](=O)[O-])c(N(C)[N+](=...
2,373.75,O=c1ccc([N+](=O)[O-])cn1Cc1ccccc1
3,373.70,N#CC(Cc1ccccc1)(Cc1ccco1)c1ccc([N+](=O)[O-])cc1
4,374.25,CCOC(=O)c1cc2ccccc2cc1-c1ccc([N+](=O)[O-])cc1
...,...,...
117747,395.65,BrC1=CC2=C(C=C1)N=C(O2)C1=CC=CC=C1N(=O)=O
117748,396.15,C[C@@H]1CC(=O)C[C@@H](N1)C1=CC=C(C=C1)N(=O)=O
117749,327.65,O=N(=O)C1=C(OCC#N)C=CC=C1
117750,488.15,O=N(=O)C1=CC(=C(NN=C\C=C\C2=CC=CO2)C=C1)N(=O)=O


In [6]:
smiles = [smi for smi in df['SMILES']]

In [7]:
len(smiles)

117752

In [23]:
%%time

clf = UniMolRepr(data_type='molecule', 
                 remove_hs=False,
                 model_name='unimolv1', # avaliable: unimolv1, unimolv2
                 model_size='84m', # work when model_name is unimolv2. avaliable: 84m, 164m, 310m, 570m, 1.1B.
                 )

smiles_list = smiles
unimol_repr = clf.get_repr(smiles_list, return_atomic_reprs=False)
embeddings = np.array(unimol_repr['cls_repr'])
print(np.array(unimol_repr['cls_repr']).shape)

2025-04-18 17:14:50 | unimol_tools/models/unimol.py | 135 | INFO | Uni-Mol Tools | Loading pretrained weights from /opt/anaconda/envs/torch_gcnn/lib/python3.10/site-packages/unimol_tools/weights/mol_pre_all_h_220816.pt
2025-04-18 17:16:14 | unimol_tools/data/conformer.py | 150 | INFO | Uni-Mol Tools | Start generating conformers...
117752it [22:08, 88.66it/s] 
2025-04-18 17:38:25 | unimol_tools/data/conformer.py | 161 | INFO | Uni-Mol Tools | Succeeded in generating conformers for 100.00% of molecules.
2025-04-18 17:38:28 | unimol_tools/data/conformer.py | 178 | INFO | Uni-Mol Tools | Succeeded in generating 3d conformers for 99.94% of molecules.
2025-04-18 17:38:28 | unimol_tools/data/conformer.py | 187 | INFO | Uni-Mol Tools | Failed 3d conformers indices: [2199, 3013, 5897, 6977, 8098, 8524, 8634, 11132, 13988, 56451, 56649, 63951, 68202, 78798, 78903, 78921, 80652, 80718, 81377, 81380, 81665, 81666, 81900, 82349, 86205, 86207, 86208, 87248, 89647, 91168, 91188, 91475, 92077, 92124,

(117752, 512)
CPU times: user 40min 11s, sys: 1min 2s, total: 41min 14s
Wall time: 31min 47s


In [24]:
emb_mol = pd.DataFrame(embeddings)

In [4]:
emb_mol = pd.read_csv('uni-mol.csv')

In [5]:
emb_mol

,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,MP
0,-0.013992,-0.669688,-0.810371,-0.820527,-0.944870,-1.545177,1.004045,0.720911,-0.538595,1.131984,...,0.365982,1.061863,-2.759452,1.193249,-0.218466,0.379487,2.335730,0.893357,-2.023531,355.15
1,-0.255941,-0.482257,-0.014340,-0.300146,-0.454813,-1.467290,0.742272,0.779284,-0.378655,1.614774,...,0.528400,0.815105,-2.774471,1.246665,0.295793,-0.397826,2.357099,0.496361,-2.069937,373.65
2,-0.116311,-0.716807,-0.596487,-0.134767,-0.437958,-1.901230,1.060897,0.929643,-0.660413,1.297815,...,0.430205,0.736438,-2.816033,1.252844,0.396341,-0.504774,2.235858,0.548843,-2.080271,373.75
3,-0.094511,-0.824312,-0.569022,-0.392049,-0.464151,-1.441643,0.799341,0.841834,-0.314347,1.312179,...,0.576383,0.337680,-2.739552,1.275348,0.565805,-0.853423,2.139378,0.804444,-2.055217,373.70
4,-0.291880,-0.746322,-0.622199,-0.409220,-0.443025,-1.259129,0.990855,0.646626,-0.346968,1.213809,...,0.606900,0.408470,-2.747625,1.222448,0.677137,-0.627984,2.115620,0.872131,-2.057241,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,-0.073738,-0.700036,-0.405322,-0.244470,-0.489630,-1.644408,0.836901,0.682566,-0.335481,1.285728,...,0.198654,0.263748,-2.825043,1.211132,0.321607,-0.647005,2.305871,0.891368,-2.107719,395.65
117748,-0.146027,-0.864437,-0.300155,-0.259133,-0.658427,-1.554542,0.857640,0.988294,-0.525199,1.286521,...,0.699391,0.257819,-2.783286,1.274370,0.355458,-0.614098,2.207493,0.802195,-2.024621,396.15
117749,0.096798,-1.041117,-0.263131,-0.502725,-0.931887,-1.409036,0.755577,0.720619,-0.723526,1.525824,...,0.449580,1.015750,-2.798421,1.329739,0.094653,-0.600728,2.345972,0.610362,-2.006141,327.65
117750,-0.367869,-0.537741,-0.650870,-0.120295,-0.444859,-1.763567,1.131350,0.718563,-0.293059,1.283509,...,0.286710,0.295653,-2.660894,1.250393,0.357552,-0.294115,2.211480,0.837793,-2.026178,488.15


In [33]:
emb_mol["MP"] = mps
emb_mol

,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,MP
0,-0.013992,-0.669688,-0.810371,-0.820527,-0.944870,-1.545177,1.004045,0.720911,-0.538595,1.131984,...,0.365982,1.061863,-2.759452,1.193249,-0.218466,0.379487,2.335730,0.893357,-2.023531,355.15
1,-0.255941,-0.482257,-0.014340,-0.300146,-0.454813,-1.467290,0.742272,0.779284,-0.378655,1.614774,...,0.528400,0.815105,-2.774471,1.246665,0.295793,-0.397826,2.357099,0.496361,-2.069937,373.65
2,-0.116311,-0.716807,-0.596487,-0.134767,-0.437958,-1.901230,1.060897,0.929643,-0.660413,1.297815,...,0.430205,0.736438,-2.816033,1.252844,0.396341,-0.504774,2.235858,0.548843,-2.080271,373.75
3,-0.094511,-0.824312,-0.569022,-0.392049,-0.464151,-1.441643,0.799341,0.841834,-0.314347,1.312179,...,0.576383,0.337680,-2.739552,1.275348,0.565804,-0.853423,2.139379,0.804444,-2.055217,373.70
4,-0.291880,-0.746322,-0.622199,-0.409220,-0.443025,-1.259129,0.990855,0.646626,-0.346968,1.213809,...,0.606900,0.408470,-2.747625,1.222448,0.677136,-0.627984,2.115620,0.872131,-2.057241,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,-0.073738,-0.700037,-0.405322,-0.244470,-0.489630,-1.644408,0.836901,0.682566,-0.335481,1.285728,...,0.198654,0.263748,-2.825043,1.211132,0.321607,-0.647005,2.305871,0.891368,-2.107719,395.65
117748,-0.146027,-0.864437,-0.300155,-0.259133,-0.658427,-1.554542,0.857640,0.988294,-0.525199,1.286521,...,0.699391,0.257819,-2.783286,1.274370,0.355458,-0.614098,2.207494,0.802195,-2.024621,396.15
117749,0.096798,-1.041117,-0.263131,-0.502725,-0.931887,-1.409036,0.755577,0.720619,-0.723526,1.525824,...,0.449580,1.015749,-2.798421,1.329739,0.094653,-0.600728,2.345972,0.610362,-2.006141,327.65
117750,-0.367869,-0.537741,-0.650870,-0.120295,-0.444859,-1.763567,1.131350,0.718563,-0.293059,1.283509,...,0.286710,0.295653,-2.660894,1.250393,0.357552,-0.294115,2.211480,0.837793,-2.026178,488.15


In [34]:
emb_mol.to_csv('uni-mol.csv', index=False)

In [7]:
from sklearn.model_selection import train_test_split
X_cls_train, X_cls_test, Y_train, Y_test = train_test_split(emb_mol.iloc[:, :-1], emb_mol.iloc[:, -1], test_size=0.2, random_state=42)

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

In [46]:
%%time
model = xgb.XGBRegressor(n_estimators=1000, n_jobs=-1, learning_rate=0.05, max_depth=7)

model.fit(X_cls_train, Y_train)

Y_pred = model.predict(X_cls_test)

mse = mean_squared_error(Y_test, Y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print(f"MSE: {mse}, RMSE: {rmse}, MAE: {mae}, R²: {r2}")

MSE: 1697.8466482617748, RMSE: 41.20493475618879, MAE: 31.828206495026574, R²: 0.6660609835707013
CPU times: user 12h 46min 58s, sys: 2min 7s, total: 12h 49min 6s
Wall time: 1h 12min 38s


In [48]:
print(f"MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.2f}")

MSE: 1697.85, RMSE: 41.20, MAE: 31.83, R²: 0.67


In [9]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Данные уже подготовлены ранее:
X = emb_mol.iloc[:, :-1]
y = emb_mol.iloc[:, -1]

# Определяем количество фолдов для кросс-валидации
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Создаем массивы для хранения результатов каждой итерации
maes = []
rmsses = []

for train_idx, val_idx in kf.split(X):
    # Разделяем выборку на тренировочную и проверочную части
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Обучение модели
    model = xgb.XGBRegressor(
        n_estimators=1000,
        n_jobs=-1,
        learning_rate=0.05,
        max_depth=7
    )
    model.fit(X_train, y_train)
    
    # Прогнозирование значений на валидной выборке
    y_pred = model.predict(X_val)
    
    # Вычисляем метрику среднеквадратичной ошибки (MSE) и извлекаем корень
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    
    # Среднеабсолютная ошибка (MAE)
    mae = mean_absolute_error(y_val, y_pred)
    
    # Сохранение результатов
    rmsses.append(rmse)
    maes.append(mae)

# Подсчет средних значений и стандартных отклонений
mean_mae = np.mean(maes)
std_mae = np.std(maes)
mean_rmse = np.mean(rmsses)
std_rmse = np.std(rmsses)

# Печать результата
print(f'MAE: {mean_mae:.4f} ± {std_mae:.4f}')
print(f'RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}')

MAE: 32.0454 ± 0.2397
RMSE: 41.5820 ± 0.3726
